## 1. Load the final test predictions

In [1]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from restaurant_risk.dataset.load import load_modeling_dataset
from restaurant_risk.dataset.splits import create_primary_temporal_splits
from restaurant_risk.modeling.features import (
    get_feature_matrix,
    get_target_vector,
)

from restaurant_risk.modeling.baselines import (
    add_smoothed_historical_risk,
)

from restaurant_risk.evaluation.metrics import (
    evaluate_ranking_policy,
    random_expected_metrics,
)

from restaurant_risk.evaluation.ranking import select_top_k

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\User\projects\restaurant-inspection-prioritization


In [2]:
DATABASE_PATH = PROJECT_ROOT / "data" / "restaurant_risk.duckdb"

df = load_modeling_dataset(DATABASE_PATH)

print("Dataset shape:", df.shape)
print(
    "Cutoff date:",
    df["cutoff_date"].min(),
    "→",
    df["cutoff_date"].max(),
)
print(
    "Target prevalence:",
    f"{df['target_high_severity'].mean():.2%}",
)

Dataset shape: (36037, 49)
Cutoff date: 2010-02-26 00:00:00 → 2026-05-29 00:00:00
Target prevalence: 29.64%


In [3]:
DATABASE_PATH = PROJECT_ROOT / "data" / "restaurant_risk.duckdb"

df = load_modeling_dataset(DATABASE_PATH)

print("Dataset shape:", df.shape)
print(
    "Cutoff date:",
    df["cutoff_date"].min(),
    "→",
    df["cutoff_date"].max(),
)
print(
    "Target prevalence:",
    f"{df['target_high_severity'].mean():.2%}",
)

Dataset shape: (36037, 49)
Cutoff date: 2010-02-26 00:00:00 → 2026-05-29 00:00:00
Target prevalence: 29.64%


In [4]:
train_df, validation_df, test_df = create_primary_temporal_splits(df)

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nDate ranges:")
for name, data in [
    ("Train", train_df),
    ("Validation", validation_df),
    ("Test", test_df),
]:
    print(
        f"{name:<11}",
        data["cutoff_date"].min(),
        "→",
        data["cutoff_date"].max(),
    )

Train: (12865, 49)
Validation: (16057, 49)
Test: (6396, 49)

Date ranges:
Train       2022-01-03 00:00:00 → 2023-12-29 00:00:00
Validation  2024-01-02 00:00:00 → 2024-12-31 00:00:00
Test        2025-01-02 00:00:00 → 2025-12-31 00:00:00


In [5]:
MODEL_PATH = PROJECT_ROOT / "models" / "logistic_regression_C1.joblib"
CALIBRATOR_PATH = (
    PROJECT_ROOT
    / "models"
    / "logistic_regression_C1_sigmoid_calibrator.joblib"
)

model = joblib.load(MODEL_PATH)
calibrator = joblib.load(CALIBRATOR_PATH)

print("Model:", type(model).__name__)
print("Calibrator:", type(calibrator).__name__)

Model: Pipeline
Calibrator: SigmoidCalibrator


In [6]:
X_test = get_feature_matrix(test_df)
y_test = get_target_vector(test_df)

test_logistic_scores = model.predict_proba(X_test)[:, 1]

test_calibrated_scores = (
    calibrator.predict_positive_probability(
        model=model,
        data_to_score=test_df,
    ).to_numpy()
)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)
print("Raw scores:", test_logistic_scores.shape)
print("Calibrated scores:", test_calibrated_scores.shape)

X_test: (6396, 21)
y_test: (6396,)
Raw scores: (6396,)
Calibrated scores: (6396,)


## 2. Error Analysis

In [7]:
error_df = test_df[
    [
        "camis",
        "cutoff_inspection_id",
        "cutoff_date",
        "borough",
        "cuisine_description",
        "history_depth_bucket",
        "has_no_cycle_initial_history",
        "prior_cycle_inspection_count",
        "prior_critical_inspection_rate",
        "prior_high_severity_inspection_rate",
        "days_since_last_cycle_inspection",
        "average_historical_score",
        "target_high_severity",
    ]
].copy()

error_df["raw_probability"] = test_logistic_scores
error_df["calibrated_probability"] = test_calibrated_scores

print("Error analysis dataset:", error_df.shape)

display(error_df.head())

Error analysis dataset: (6396, 15)


,camis,cutoff_inspection_id,cutoff_date,borough,cuisine_description,history_depth_bucket,has_no_cycle_initial_history,prior_cycle_inspection_count,prior_critical_inspection_rate,prior_high_severity_inspection_rate,days_since_last_cycle_inspection,average_historical_score,target_high_severity,raw_probability,calibrated_probability
4,41047349,1ae60b760d3f8a6c7883e2d4d12c524f,2025-01-14,Manhattan,Latin American,2-3,False,2,1.0,0.000000,0,10.500000,0,0.155675,0.189148
5,50119840,1d8bbc92d6eb5528aeee73abfa4b57c7,2025-07-15,Brooklyn,Chinese,2-3,False,4,1.0,0.250000,0,24.000000,1,0.275364,0.302659
7,50117697,31bcc46cd7e9e01ef0be02608e8fc04c,2025-04-03,Queens,Chinese,2-3,False,4,1.0,0.750000,0,48.250000,0,0.967068,0.947734
9,40530630,42b1029ebef12a93b3772fdde5b44542,2025-04-09,Bronx,Bakery Products/Desserts,2-3,False,5,1.0,0.000000,0,15.400000,0,0.142751,0.176099
25,50092484,be647794f9474dfe4282956f272b17e9,2025-05-12,Brooklyn,Spanish,2-3,False,3,1.0,0.666667,0,28.333333,1,0.163624,0.197075


In [8]:
print("Target prevalence:")
print(f"{error_df['target_high_severity'].mean():.2%}")

print("\nCalibrated probability distribution:")
display(
    error_df["calibrated_probability"].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    ).to_frame()
)

Target prevalence:
37.20%

Calibrated probability distribution:


,calibrated_probability
count,6396.000000
mean,0.298756
std,0.178096
min,0.013169
1%,0.050671
5%,0.085276
10%,0.105225
25%,0.155019
50%,0.261302
75%,0.404312


In [9]:
THRESHOLD = 0.50

error_df["predicted_positive"] = (
    error_df["calibrated_probability"] >= THRESHOLD
).astype(int)

error_df["error_type"] = np.select(
    [
        (error_df["predicted_positive"] == 1)
        & (error_df["target_high_severity"] == 1),

        (error_df["predicted_positive"] == 1)
        & (error_df["target_high_severity"] == 0),

        (error_df["predicted_positive"] == 0)
        & (error_df["target_high_severity"] == 1),

        (error_df["predicted_positive"] == 0)
        & (error_df["target_high_severity"] == 0),
    ],
    [
        "True Positive",
        "False Positive",
        "False Negative",
        "True Negative",
    ],
    default="Unknown",
)

display(
    error_df["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .to_frame("count")
)

,count
error_type,
True Negative,3676
False Negative,1801
True Positive,578
False Positive,341


In [10]:
false_positives = (
    error_df[error_df["error_type"] == "False Positive"]
    .sort_values("calibrated_probability", ascending=False)
)

print("False positives:", len(false_positives))

display(
    false_positives[
        [
            "camis",
            "cutoff_date",
            "borough",
            "cuisine_description",
            "history_depth_bucket",
            "prior_cycle_inspection_count",
            "prior_critical_inspection_rate",
            "prior_high_severity_inspection_rate",
            "calibrated_probability",
        ]
    ].head(20)
)

False positives: 341


,camis,cutoff_date,borough,cuisine_description,history_depth_bucket,prior_cycle_inspection_count,prior_critical_inspection_rate,prior_high_severity_inspection_rate,calibrated_probability
7,50117697,2025-04-03,Queens,Chinese,2-3,4,1.0,0.750000,0.947734
19593,50117697,2025-03-10,Queens,Chinese,2-3,3,1.0,1.000000,0.935496
31244,50044289,2025-03-12,Manhattan,Chinese,2-3,4,1.0,1.000000,0.881278
6664,50100141,2025-03-14,Queens,Caribbean,2-3,5,1.0,0.800000,0.865858
26709,41387224,2025-09-22,Manhattan,African,2-3,4,1.0,1.000000,0.857351
21067,41686215,2025-12-31,Queens,Chinese,4+,7,1.0,1.000000,0.839913
4807,50072230,2025-11-10,Queens,Japanese,2-3,6,1.0,1.000000,0.827270
14287,50121526,2025-03-31,Manhattan,African,2-3,3,1.0,1.000000,0.818406
10539,50001241,2025-01-09,Manhattan,Bangladeshi,2-3,4,1.0,1.000000,0.813144
9982,50075044,2025-06-09,Brooklyn,African,2-3,3,1.0,0.666667,0.812692


In [11]:
false_negatives = (
    error_df[error_df["error_type"] == "False Negative"]
    .sort_values("calibrated_probability", ascending=True)
)

print("False negatives:", len(false_negatives))

display(
    false_negatives[
        [
            "camis",
            "cutoff_date",
            "borough",
            "cuisine_description",
            "history_depth_bucket",
            "prior_cycle_inspection_count",
            "prior_critical_inspection_rate",
            "prior_high_severity_inspection_rate",
            "calibrated_probability",
        ]
    ].head(20)
)

False negatives: 1801


,camis,cutoff_date,borough,cuisine_description,history_depth_bucket,prior_cycle_inspection_count,prior_critical_inspection_rate,prior_high_severity_inspection_rate,calibrated_probability
334,41333188,2025-04-02,Manhattan,Coffee/Tea,4+,7,0.857143,0.428571,0.040138
25554,50009827,2025-09-04,Brooklyn,Bagels/Pretzels,2-3,4,1.000000,0.000000,0.048096
9280,50048295,2025-03-11,Brooklyn,Donuts,2-3,3,0.666667,0.000000,0.050954
28977,50118094,2025-09-11,Brooklyn,Donuts,2-3,2,0.000000,0.000000,0.052097
26916,50055939,2025-09-22,Queens,Donuts,2-3,2,0.000000,0.000000,0.056772
33569,41257749,2025-04-14,Staten Island,Chicken,4+,7,0.857143,0.142857,0.057981
21934,40808118,2025-07-31,0,French,2-3,4,1.000000,1.000000,0.063605
15128,41306801,2025-04-01,Bronx,Donuts,2-3,2,0.000000,0.000000,0.064107
19334,50016526,2025-09-12,Queens,Sandwiches,4+,6,1.000000,0.333333,0.065904
9685,50009827,2025-10-17,Brooklyn,Bagels/Pretzels,2-3,5,1.000000,0.000000,0.066369


In [12]:
high_confidence_false_positives = error_df[
    (error_df["calibrated_probability"] >= 0.80)
    & (error_df["target_high_severity"] == 0)
].sort_values(
    "calibrated_probability",
    ascending=False,
)

high_confidence_false_negatives = error_df[
    (error_df["calibrated_probability"] <= 0.20)
    & (error_df["target_high_severity"] == 1)
].sort_values(
    "calibrated_probability",
    ascending=True,
)

print(
    "High-confidence false positives:",
    len(high_confidence_false_positives),
)

print(
    "High-confidence false negatives:",
    len(high_confidence_false_negatives),
)

High-confidence false positives: 12
High-confidence false negatives: 488


In [13]:
error_columns = [
    "camis",
    "cutoff_date",
    "borough",
    "cuisine_description",
    "history_depth_bucket",
    "prior_cycle_inspection_count",
    "prior_critical_inspection_rate",
    "prior_high_severity_inspection_rate",
    "days_since_last_cycle_inspection",
    "average_historical_score",
    "calibrated_probability",
]

print("HIGH-CONFIDENCE FALSE POSITIVES")
display(
    high_confidence_false_positives[error_columns].head(20)
)

print("\nHIGH-CONFIDENCE FALSE NEGATIVES")
display(
    high_confidence_false_negatives[error_columns].head(20)
)

HIGH-CONFIDENCE FALSE POSITIVES


,camis,cutoff_date,borough,cuisine_description,history_depth_bucket,prior_cycle_inspection_count,prior_critical_inspection_rate,prior_high_severity_inspection_rate,days_since_last_cycle_inspection,average_historical_score,calibrated_probability
7,50117697,2025-04-03,Queens,Chinese,2-3,4,1.0,0.750000,0,48.250000,0.947734
19593,50117697,2025-03-10,Queens,Chinese,2-3,3,1.0,1.000000,0,60.000000,0.935496
31244,50044289,2025-03-12,Manhattan,Chinese,2-3,4,1.0,1.000000,0,41.750000,0.881278
6664,50100141,2025-03-14,Queens,Caribbean,2-3,5,1.0,0.800000,0,46.000000,0.865858
26709,41387224,2025-09-22,Manhattan,African,2-3,4,1.0,1.000000,0,52.250000,0.857351
21067,41686215,2025-12-31,Queens,Chinese,4+,7,1.0,1.000000,0,49.285714,0.839913
4807,50072230,2025-11-10,Queens,Japanese,2-3,6,1.0,1.000000,0,42.000000,0.827270
14287,50121526,2025-03-31,Manhattan,African,2-3,3,1.0,1.000000,0,46.000000,0.818406
10539,50001241,2025-01-09,Manhattan,Bangladeshi,2-3,4,1.0,1.000000,0,42.250000,0.813144
9982,50075044,2025-06-09,Brooklyn,African,2-3,3,1.0,0.666667,0,34.000000,0.812692



HIGH-CONFIDENCE FALSE NEGATIVES


,camis,cutoff_date,borough,cuisine_description,history_depth_bucket,prior_cycle_inspection_count,prior_critical_inspection_rate,prior_high_severity_inspection_rate,days_since_last_cycle_inspection,average_historical_score,calibrated_probability
334,41333188,2025-04-02,Manhattan,Coffee/Tea,4+,7,0.857143,0.428571,0,17.142857,0.040138
25554,50009827,2025-09-04,Brooklyn,Bagels/Pretzels,2-3,4,1.000000,0.000000,0,21.500000,0.048096
9280,50048295,2025-03-11,Brooklyn,Donuts,2-3,3,0.666667,0.000000,0,7.333333,0.050954
28977,50118094,2025-09-11,Brooklyn,Donuts,2-3,2,0.000000,0.000000,0,4.500000,0.052097
26916,50055939,2025-09-22,Queens,Donuts,2-3,2,0.000000,0.000000,0,3.500000,0.056772
33569,41257749,2025-04-14,Staten Island,Chicken,4+,7,0.857143,0.142857,0,17.714286,0.057981
21934,40808118,2025-07-31,0,French,2-3,4,1.000000,1.000000,0,29.250000,0.063605
15128,41306801,2025-04-01,Bronx,Donuts,2-3,2,0.000000,0.000000,0,6.000000,0.064107
19334,50016526,2025-09-12,Queens,Sandwiches,4+,6,1.000000,0.333333,0,24.833333,0.065904
9685,50009827,2025-10-17,Brooklyn,Bagels/Pretzels,2-3,5,1.000000,0.000000,0,18.600000,0.066369


## 3. History-Depth Evaluation

In [14]:
history_summary = (
    error_df
    .groupby("history_depth_bucket", observed=False)
    .agg(
        observations=("target_high_severity", "size"),
        positives=("target_high_severity", "sum"),
        prevalence=("target_high_severity", "mean"),
    )
    .reset_index()
)

display(history_summary)

,history_depth_bucket,observations,positives,prevalence
0,1,2412,1066,0.441957
1,2-3,3941,1301,0.330119
2,4+,43,12,0.279070


In [15]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
)

history_metrics = []

for history_bucket, group in error_df.groupby(
    "history_depth_bucket",
    observed=False,
):
    y = group["target_high_severity"].to_numpy()
    scores = group["calibrated_probability"].to_numpy()

    metrics = {
        "history_depth": history_bucket,
        "n": len(group),
        "positive_rate": y.mean(),
        "average_precision": average_precision_score(y, scores),
        "brier_score": brier_score_loss(y, scores),
    }

    # ROC-AUC requires both classes to be present
    if len(np.unique(y)) == 2:
        metrics["roc_auc"] = roc_auc_score(y, scores)
    else:
        metrics["roc_auc"] = np.nan

    history_metrics.append(metrics)

history_metrics_df = pd.DataFrame(history_metrics)

display(
    history_metrics_df.sort_values("history_depth")
)

,history_depth,n,positive_rate,average_precision,brier_score,roc_auc
0,1,2412,0.441957,0.564071,0.239439,0.635402
1,2-3,3941,0.330119,0.557279,0.200110,0.709902
2,4+,43,0.279070,0.421830,0.227606,0.599462


In [16]:
test_with_baseline = add_smoothed_historical_risk(
    train_df=train_df,
    data_to_score=test_df,
    alpha=5.0,
    score_column="smoothed_historical_high_severity_risk",
)

print("Test rows:", len(test_with_baseline))

display(
    test_with_baseline[
        [
            "camis",
            "target_high_severity",
            "smoothed_historical_high_severity_risk",
        ]
    ].head()
)

Test rows: 6396


,camis,target_high_severity,smoothed_historical_high_severity_risk
4,41047349,0,0.197601
5,50119840,1,0.264801
7,50117697,0,0.487023
9,40530630,0,0.138321
25,50092484,1,0.422901


In [17]:
baseline_scores = (
    test_with_baseline[
        "smoothed_historical_high_severity_risk"
    ].to_numpy()
)

assert len(baseline_scores) == len(test_df)
assert np.isfinite(baseline_scores).all()
assert ((baseline_scores >= 0) & (baseline_scores <= 1)).all()

print("Historical-risk baseline sanity checks passed.")
print()
print(pd.Series(baseline_scores).describe())

Historical-risk baseline sanity checks passed.

count    6396.000000
mean        0.317786
std         0.116658
min         0.125746
25%         0.197601
50%         0.297901
75%         0.397202
max         0.698601
dtype: float64


In [18]:
error_df["historical_risk"] = baseline_scores

display(
    error_df[
        [
            "camis",
            "history_depth_bucket",
            "target_high_severity",
            "historical_risk",
            "calibrated_probability",
        ]
    ].head(10)
)

,camis,history_depth_bucket,target_high_severity,historical_risk,calibrated_probability
4,41047349,2-3,0,0.197601,0.189148
5,50119840,2-3,1,0.264801,0.302659
7,50117697,2-3,0,0.487023,0.947734
9,40530630,2-3,0,0.138321,0.176099
25,50092484,2-3,1,0.422901,0.197075
45,40818115,2-3,0,0.197601,0.101925
47,50156239,1,1,0.397202,0.407177
56,50112504,2-3,1,0.487023,0.675311
63,50110783,2-3,0,0.297901,0.134769
65,50081134,1,0,0.197601,0.218340


In [19]:
comparison_metrics = []

for history_bucket, group in error_df.groupby(
    "history_depth_bucket",
    observed=False,
):
    y = group["target_high_severity"].to_numpy()

    historical_scores = group["historical_risk"].to_numpy()
    ml_scores = group["calibrated_probability"].to_numpy()

    row = {
        "history_depth": history_bucket,
        "n": len(group),
        "prevalence": y.mean(),
        "historical_AP": average_precision_score(y, historical_scores),
        "ML_AP": average_precision_score(y, ml_scores),
        "historical_ROC_AUC": roc_auc_score(y, historical_scores),
        "ML_ROC_AUC": roc_auc_score(y, ml_scores),
        "historical_Brier": brier_score_loss(y, historical_scores),
        "ML_Brier": brier_score_loss(y, ml_scores),
    }

    row["AP_improvement"] = row["ML_AP"] - row["historical_AP"]
    row["ROC_AUC_improvement"] = (
        row["ML_ROC_AUC"] - row["historical_ROC_AUC"]
    )
    row["Brier_improvement"] = (
        row["historical_Brier"] - row["ML_Brier"]
    )

    comparison_metrics.append(row)

history_comparison_df = pd.DataFrame(comparison_metrics)

display(history_comparison_df)

,history_depth,n,prevalence,historical_AP,ML_AP,historical_ROC_AUC,ML_ROC_AUC,historical_Brier,ML_Brier,AP_improvement,ROC_AUC_improvement,Brier_improvement
0,1,2412,0.441957,0.526148,0.564071,0.607272,0.635402,0.252077,0.239439,0.037923,0.028129,0.012638
1,2-3,3941,0.330119,0.525069,0.557279,0.692168,0.709902,0.197325,0.200110,0.032209,0.017734,-0.002785
2,4+,43,0.279070,0.293218,0.421830,0.509409,0.599462,0.240437,0.227606,0.128612,0.090054,0.012831


In [20]:
policy_df = error_df[
    [
        "camis",
        "target_high_severity",
        "historical_risk",
        "calibrated_probability",
    ]
].copy()

print("Policy evaluation population:", len(policy_df))
print("Total high-severity outcomes:", policy_df["target_high_severity"].sum())

Policy evaluation population: 6396
Total high-severity outcomes: 2379


In [21]:
capacities = [50, 100, 250, 500]

policy_results = []

total_positives = policy_df["target_high_severity"].sum()
population_size = len(policy_df)

for capacity in capacities:

    # Historical-risk policy
    historical_top = (
        policy_df
        .sort_values(
            ["historical_risk", "camis"],
            ascending=[False, True],
        )
        .head(capacity)
    )

    # ML policy
    ml_top = (
        policy_df
        .sort_values(
            ["calibrated_probability", "camis"],
            ascending=[False, True],
        )
        .head(capacity)
    )

    historical_yield = historical_top["target_high_severity"].sum()
    ml_yield = ml_top["target_high_severity"].sum()

    random_expected = (
        capacity
        * total_positives
        / population_size
    )

    policy_results.append(
        {
            "capacity": capacity,
            "random_expected_yield": random_expected,
            "historical_yield": historical_yield,
            "ML_yield": ml_yield,
            "historical_precision": historical_yield / capacity,
            "ML_precision": ml_yield / capacity,
            "ML_incremental_yield_vs_historical": (
                ml_yield - historical_yield
            ),
        }
    )

policy_results_df = pd.DataFrame(policy_results)

display(policy_results_df)

,capacity,random_expected_yield,historical_yield,ML_yield,historical_precision,ML_precision,ML_incremental_yield_vs_historical
0,50,18.597561,33,44,0.660,0.880,11
1,100,37.195122,72,81,0.720,0.810,9
2,250,92.987805,171,188,0.684,0.752,17
3,500,185.975610,324,333,0.648,0.666,9


In [22]:
policy_results_df["historical_lift_vs_random"] = (policy_results_df["historical_yield"]/ policy_results_df["random_expected_yield"])
policy_results_df["ML_lift_vs_random"] = (policy_results_df["ML_yield"]/ policy_results_df["random_expected_yield"])
policy_results_df["ML_relative_improvement_vs_historical"] = (policy_results_df["ML_yield"]/ policy_results_df["historical_yield"]- 1)
display(policy_results_df)

,capacity,random_expected_yield,historical_yield,ML_yield,historical_precision,ML_precision,ML_incremental_yield_vs_historical,historical_lift_vs_random,ML_lift_vs_random,ML_relative_improvement_vs_historical
0,50,18.597561,33,44,0.660,0.880,11,1.774426,2.365902,0.333333
1,100,37.195122,72,81,0.720,0.810,9,1.935738,2.177705,0.125000
2,250,92.987805,171,188,0.684,0.752,17,1.838951,2.021770,0.099415
3,500,185.975610,324,333,0.648,0.666,9,1.742164,1.790557,0.027778


In [23]:
overlap_results = []

for capacity in capacities:
    historical_top = set(
        policy_df
        .sort_values(
            ["historical_risk", "camis"],
            ascending=[False, True],
        )
        .head(capacity)["camis"]
    )

    ml_top = set(
        policy_df
        .sort_values(
            ["calibrated_probability", "camis"],
            ascending=[False, True],
        )
        .head(capacity)["camis"]
    )

    overlap = historical_top & ml_top

    overlap_results.append(
        {
            "capacity": capacity,
            "historical_queue_size": len(historical_top),
            "ML_queue_size": len(ml_top),
            "overlap": len(overlap),
            "overlap_rate": len(overlap) / capacity,
            "ML_only": len(ml_top - historical_top),
            "historical_only": len(historical_top - ml_top),
        }
    )

overlap_df = pd.DataFrame(overlap_results)

display(overlap_df)

,capacity,historical_queue_size,ML_queue_size,overlap,overlap_rate,ML_only,historical_only
0,50,29,35,7,0.140,28,22
1,100,71,70,19,0.190,51,52
2,250,177,177,57,0.228,120,120
3,500,318,347,145,0.290,202,173


In [24]:
capacity = 100

historical_top_100 = (
    policy_df
    .sort_values(
        ["historical_risk", "camis"],
        ascending=[False, True],
    )
    .head(capacity)
)

ml_top_100 = (
    policy_df
    .sort_values(
        ["calibrated_probability", "camis"],
        ascending=[False, True],
    )
    .head(capacity)
)

historical_ids = set(historical_top_100["camis"])
ml_ids = set(ml_top_100["camis"])

ml_only_100 = policy_df[
    policy_df["camis"].isin(ml_ids - historical_ids)
].sort_values(
    "calibrated_probability",
    ascending=False,
)

display(
    ml_only_100[
        [
            "camis",
            "target_high_severity",
            "historical_risk",
            "calibrated_probability",
        ]
    ]
)

,camis,target_high_severity,historical_risk,calibrated_probability
9528,50111296,1,0.487023,0.964713
7,50117697,0,0.487023,0.947734
33371,50138270,1,0.487023,0.942635
31264,50056245,1,0.547901,0.941386
19593,50117697,0,0.547901,0.935496
...,...,...,...,...
30948,50112504,1,0.422901,0.449876
18526,50138117,1,0.230535,0.436456
23534,50072911,1,0.483316,0.417110
25714,41670224,1,0.422901,0.406289


# 5. Conclusions and Limitations

## Conclusion

The frozen logistic regression model provides incremental value over the
historical-risk prioritization baseline on the untouched 2025 temporal test
set.

Under identical inspection capacities, the ML policy identifies more
high-severity outcomes than historical-risk prioritization:

| Inspection Capacity | Random Expected | Historical Risk | ML | ML Increment |
|---:|---:|---:|---:|---:|
| 50  | 18.6 | 33  | 44  | +11 |
| 100 | 37.2 | 72  | 81  | +9  |
| 250 | 93.0 | 171 | 188 | +17 |
| 500 | 186.0 | 324 | 333 | +9  |

The ML advantage is largest at lower inspection capacities and narrows as
capacity increases, consistent with increasing overlap between the two
prioritization policies.

History-depth analysis shows that performance is strongest and most stable
for restaurants with 2–3 prior inspections. The 4+ cohort contains only 43
test observations and therefore does not support a strong conclusion about
performance at that history depth.

Error analysis also identifies a meaningful limitation: false negatives,
including high-confidence false negatives, remain substantial. The model
should therefore be treated as a prioritization mechanism rather than a
perfect classifier.

## Operational Interpretation

The model estimates the probability of a high-severity outcome at the next
eligible routine-cycle initial inspection. It does not estimate whether a
restaurant is currently "safe" or "unsafe", and the analysis does not make
causal claims about inspection outcomes.

The scoring population represents a reproducible set of restaurants that can
be scored from the available source data. It should not be interpreted as the
city's actual operational due-for-inspection queue.

## Limitations

- The 4+ history-depth cohort is very small.
- False negatives remain a significant limitation.
- The model is trained and evaluated using data from one city and one
  inspection system.
- The target represents an administrative inspection outcome rather than
  an underlying latent measure of restaurant safety.
- The available public data does not expose the complete operational
  inspection queue or all factors used by inspectors.
- The policy experiment is retrospective; it does not establish causal
  effects of changing inspection allocation.
- Model performance may change as inspection practices, restaurant
  populations, or data-generation processes change.

## Final Decision

The frozen logistic regression model and sigmoid calibrator are accepted as
the production candidate.

No further model tuning will be performed on the current temporal test set.

Phase A is complete. The project proceeds to Phase B: productionization,
batch scoring, API serving, testing, containerization, CI/CD, monitoring,
and deployment.